In [2]:
import os
import time
from datetime import datetime, timedelta
import pandas as pd
from dotenv import load_dotenv
from selenium import webdriver
from selenium.webdriver.edge.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException, WebDriverException
from selenium.webdriver.edge.options import Options


In [3]:
load_dotenv()
USERNAME = os.getenv("CLOUD_USERNAME")
PASSWORD = os.getenv("CLOUD_PASSWORD")

In [4]:
BASE_DIR = os.getenv("BASE_DIR", os.getcwd())
print(f"Base directory: {BASE_DIR}")

Base directory: d:\solar-performance-monitoring\notebooks


In [5]:

# -------------------------------
# Dynamic Base Path (Outer data folder)
# -------------------------------
BASE_DIR = os.getenv(
    "BASE_DIR",
    os.path.join(os.path.dirname(os.getcwd()), "data")
)

RAW_DATA_DIR = os.path.join(BASE_DIR, "raw/error_files")
POINTER_FILE = os.path.join(BASE_DIR, "date_pointer.txt")

os.makedirs(RAW_DATA_DIR, exist_ok=True)

In [6]:
# -------------------------------
# Pointer Functions
# -------------------------------
def get_start_date():
    if os.path.exists(POINTER_FILE):
        try:
            with open(POINTER_FILE, "r") as f:
                return datetime.strptime(f.read().strip(), "%Y-%m-%d")
        except:
            pass



def save_pointer(date_obj):
    temp_file = POINTER_FILE + ".tmp"
    with open(temp_file, "w") as f:
        f.write(date_obj.strftime("%Y-%m-%d"))
    os.replace(temp_file, POINTER_FILE)

In [24]:

# -------------------------------
# Edge download preferences
# -------------------------------
edge_options = Options()
edge_options.add_experimental_option(
    "prefs",
    {
        "download.default_directory": RAW_DATA_DIR,
        "download.prompt_for_download": False,
        "download.directory_upgrade": True,
        "safebrowsing.enabled": True
    }
)
edge_options.add_argument("--disable-notifications")


# -------------------------------
# Driver setup
# -------------------------------
try:

    driver = webdriver.Edge(options=edge_options)
    driver.maximize_window()
except WebDriverException as e:
    print("❌ Error initializing Edge WebDriver:", e)
    exit(1)

# time.sleep(10)
wait = WebDriverWait(driver, 20)

# -------------------------------
# Open login page
# -------------------------------
try:
    driver.get("https://www.cloudinverter.net/dist/#/logs")
except WebDriverException as e:
    print("❌ Error loading page:", e)
    driver.quit()
    exit(1)

# -------------------------------
# Login
# -------------------------------
try:
    username_input = wait.until(
        EC.presence_of_element_located((By.XPATH,
        '/html/body/div/div/div[2]/div[2]/div/div/div[2]/div/div/form/div[1]/div/div[2]/div/div/input'))
    )
    username_input.send_keys(USERNAME)

    password_input = driver.find_element(
        By.XPATH,
        '/html/body/div/div/div[2]/div[2]/div/div/div[2]/div/div/form/div[2]/div/div[2]/div/div/span/input'
    )
    password_input.send_keys(PASSWORD)

    login_btn = driver.find_element(
        By.XPATH,
        '/html/body/div/div/div[2]/div[2]/div/div/div[2]/div/div/form/div[4]/div/div/div/div/button[1]'
    )
    login_btn.click()

    # wait for logs menu (login success indicator)
    log_menu = wait.until(
        EC.element_to_be_clickable((By.XPATH,
        "/html/body/div/div/div/div[2]/div/header[2]/div/div/div[2]/ul/li[2]/span/div/div"))
    )
    log_menu.click()

except (TimeoutException, NoSuchElementException) as e:
    print("❌ Login failed:", e)
    driver.quit()
    exit(1)
# -------------------------------
# Date loop (UPDATED LOGIC)
# -------------------------------
# -------------------------------
# Date loop (UPDATED LOGIC)
# -------------------------------
start_date = get_start_date()
today = datetime.today().date()

while start_date.date() <= today:
    try:
        # 👉 start date
        start_str = start_date.strftime('%Y-%m-%d')

        # 👉 7-day range
        end_date = start_date + timedelta(days=6)

        # 👉 don't exceed today
        if end_date.date() > today:
            end_date = datetime.combine(today, datetime.min.time())

        end_str = end_date.strftime('%Y-%m-%d')

        # Open date picker
        date_range = wait.until(
            EC.element_to_be_clickable((By.CLASS_NAME, 'ant-picker-input'))
        )
        date_range.click()

        # Start date
        start_input = wait.until(
            EC.presence_of_element_located((By.XPATH,
            '/html/body/div[1]/div/div/div[2]/div/main/div/div/div[2]/div/div/div/div/div/div/div[1]/div/div/div[1]/div/div[2]/div/div[2]/div[1]/div/div[1]/input'))
        )
        start_input.clear()
        start_input.send_keys(start_str)
        start_input.send_keys(Keys.ENTER)

        # End date
        end_input = driver.find_element(
            By.XPATH,
            '/html/body/div[1]/div/div/div[2]/div/main/div/div/div[2]/div/div/div/div/div/div/div[1]/div/div/div[1]/div/div[2]/div/div[2]/div[1]/div/div[3]/input'
        )
        end_input.clear()
        end_input.send_keys(end_str)
        end_input.send_keys(Keys.ENTER)

        # Confirm
        ok_btn = driver.find_element(
            By.XPATH,
            '/html/body/div/div/div/div[2]/div/main/div/div/div[2]/div/div/div/div/div/div/div[1]/div/div/div[1]/div/div[2]/div/div[2]/div[4]/button'
        )
        ok_btn.click()

        print(f"✅ Processed {start_str} → {end_str}")

        # ✅ SAVE POINTER (IMPORTANT FIX)
        save_pointer(end_date)

        # 👉 move to next batch
        start_date = end_date + timedelta(days=1)

    except Exception as e:
        print(f"⚠️ Error for {start_date.date()}: {e}")
        break   # retry same batch next run

✅ Processed 2026-02-02 → 2026-02-08
✅ Processed 2026-02-09 → 2026-02-15
✅ Processed 2026-02-16 → 2026-02-22
✅ Processed 2026-02-23 → 2026-03-01
✅ Processed 2026-03-02 → 2026-03-08
✅ Processed 2026-03-09 → 2026-03-15
✅ Processed 2026-03-16 → 2026-03-22
✅ Processed 2026-03-23 → 2026-03-29
✅ Processed 2026-03-30 → 2026-04-05
